# 📚 Smart Research Assistant — RAG-based Knowledge System

A step-by-step walkthrough of a Retrieval-Augmented Generation (RAG) pipeline:

**PDF → chunks → embeddings → FAISS vector store → retrieval → LLM answer → evaluation**

**Stack used in this notebook:**
- **LangChain** — orchestration
- **Hugging Face** (`sentence-transformers/all-MiniLM-L6-v2`) — free, local embeddings
- **FAISS** — local vector similarity search
- **Groq** (Llama 3.1 8B) — free, fast hosted LLM for generation
- **RAGAs** — automated evaluation (faithfulness, relevance, context precision)

> **Before running:** copy `.env.example` to `.env` and add a free Groq API key from
> https://console.groq.com/keys (no OpenAI key needed anywhere in this project).


## 0. Setup — install dependencies
Run this once. Restart the kernel afterwards if this is the first install.

In [ ]:
!pip install -q -r requirements.txt

## 1. Load environment variables
This reads `GROQ_API_KEY` from your `.env` file. The key is never hardcoded or printed.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("GROQ_API_KEY"), (
    "GROQ_API_KEY not found. Copy .env.example to .env and add your key "
    "from https://console.groq.com/keys"
)
print("Groq API key loaded ✔")

## 2. Data Ingestion
Load the sample HR policy PDF (in `data/`). Swap this path for any PDF or TXT file.

In [ ]:
from rag_pipeline import load_document

FILE_PATH = "data/sample_hr_policy.pdf"

documents = load_document(FILE_PATH)
print(f"Loaded {len(documents)} page(s)")
print("\n--- Preview of page 1 ---\n")
print(documents[0].page_content[:500])

## 3. Chunking
Split the document into overlapping chunks so retrieval can find precise, relevant passages
instead of whole pages. `chunk_size=500`, `chunk_overlap=50` are good defaults for policy-style text.

In [ ]:
from rag_pipeline import chunk_documents

chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)
print(f"Split into {len(chunks)} chunks")
print("\n--- Example chunk ---\n")
print(chunks[2].page_content)

## 4. Embeddings
Convert each chunk into a vector using a free, local Hugging Face sentence-transformer model.
No API key or internet call to a paid provider is required for this step — the model downloads
once from Hugging Face and then runs locally.

In [ ]:
from rag_pipeline import get_embeddings

embeddings = get_embeddings()

# Quick sanity check: embed one chunk and inspect the vector shape
sample_vector = embeddings.embed_query(chunks[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")

## 5. Vector Storage (FAISS)
Store all chunk embeddings in a FAISS index for fast similarity search.
We also save it to disk so it can be reloaded later without re-embedding.

In [ ]:
from rag_pipeline import build_vectorstore, save_vectorstore

vectorstore = build_vectorstore(chunks, embeddings)
save_vectorstore(vectorstore, "faiss_index")
print("FAISS index built and saved to ./faiss_index")

## 6. Query Processing — test retrieval alone
Before adding the LLM, sanity-check that retrieval alone pulls back the right chunks
for a sample query.

In [ ]:
query = "What is the maternity leave policy?"

retrieved = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(retrieved, 1):
    print(f"--- Retrieved chunk {i} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content)
    print()

## 7. Answer Generation
Wire the retriever to a Groq-hosted Llama 3.1 model via a `RetrievalQA` chain.
The prompt instructs the model to answer **only** from retrieved context — this is what
makes the system "grounded" instead of a generic chatbot.

In [ ]:
from rag_pipeline import build_qa_chain, answer_question

qa_chain = build_qa_chain(vectorstore, k=3)

answer, sources = answer_question(qa_chain, query)

print("QUESTION:", query)
print("\nANSWER:\n", answer)
print(f"\nGrounded in {len(sources)} source chunk(s)")

## 8. Try more example questions
The example scenario from the project brief: upload HR policy docs, then ask natural questions.

In [ ]:
example_questions = [
    "What is the leave policy?",
    "How many days of sick leave do employees get?",
    "What is the notice period during probation?",
    "Can employees work from home?",
]

for q in example_questions:
    ans, srcs = answer_question(qa_chain, q)
    print(f"Q: {q}")
    print(f"A: {ans}")
    print("-" * 80)

## 9. Evaluation with RAGAs
Score one of the answers above on:
- **Faithfulness** — is the answer actually supported by the retrieved context (no hallucination)?
- **Answer Relevancy** — does the answer address the question asked?

Scores range 0–1, higher is better.

In [ ]:
from evaluation import evaluate_answer

question = "What is the maternity leave policy?"
answer, sources = answer_question(qa_chain, question)
context_texts = [doc.page_content for doc in sources]

scores_df = evaluate_answer(question, answer, context_texts)
scores_df

## 10. Multi-turn conversational memory (optional extension)
A simple example of adding chat history so follow-up questions ("what about for the third
child?") can be understood in context of the previous question.

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from rag_pipeline import get_llm

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=get_llm(),
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
)

result1 = conversational_chain.invoke({"question": "What is the maternity leave policy?"})
print("Q1:", result1["answer"])

result2 = conversational_chain.invoke({"question": "What about for the third child?"})
print("\nQ2 (follow-up):", result2["answer"])

## 11. Summary

This notebook demonstrated the full RAG pipeline end to end:

1. **Ingestion** — loaded a PDF and split it into overlapping chunks
2. **Embedding** — converted chunks to vectors with a free Hugging Face model
3. **Storage** — indexed vectors in FAISS for fast similarity search
4. **Retrieval + Generation** — retrieved relevant chunks and generated grounded answers with Groq's Llama 3.1
5. **Evaluation** — scored answer quality with RAGAs (faithfulness, relevancy)
6. **Memory** — extended the chain to support multi-turn conversation

For the interactive version of this same pipeline, run:
```
streamlit run app.py
```
